# 1. CV analysis

In [ ]:
import importlib
import os
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
import MixedEffectsModeling.config as config
importlib.reload(config)
from viz_style import apply_style
apply_style()

CV = Path('./CV_Results_mixed')
ENG = Path('./engine_state_mixed')
FIG = CV / 'Figures'
FIG.mkdir(parents=True, exist_ok=True)

# Reference values a correctly-specified normative model must hit on held-out HC (a true null).
TARGET = {'mean_z': 0.0, 'std_z': 1.0, 'exceed': 0.05, 'med_z2': 0.4549}  # med_z2 = median of chi2(df=1)

NZ_BINS = [24, 30, 40, 50, 60, 80, 100, 150, 200, 300, 676]
NZ_LABELS = ['25-30', '31-40', '41-50', '51-60', '61-80', '81-100', '101-150', '151-200', '201-300', '301+']

In [ ]:
# 0. Load CV artifacts
fold = pd.read_csv(CV / 'fold_stats.csv')
pool_fold = pd.read_csv(CV / 'pool_fold_stats.csv')
cv = pd.read_csv(CV / 'cv_stats.csv')
train = pd.read_csv(ENG / 'training_summary.csv')

# per-gene fold stability: a gene is "stable" only if the individual GLMM converged in all 5 folds
stab = fold.groupby('gene')['ok'].sum().rename('n_fold_ok').reset_index()
cv = cv.merge(stab, on='gene', how='left')
cv['stable'] = cv['n_fold_ok'] == config.SPIKE_PARAMS['n_splits']
cv['nz_bin'] = pd.cut(cv['nz'], bins=NZ_BINS, labels=NZ_LABELS)
cv['route_stage'] = cv['route'] + '/' + cv['stage']

print(f"engine ok genes: {int(train['ok'].sum())} | cv_stats rows: {len(cv)} "
      f"(missing {int(train['ok'].sum()) - len(cv)}: all folds failed)")
print(f"model route: {int((cv.route == 'model').sum())} | pool route: {int((cv.route == 'pool').sum())}")
print(f"\npool-route fold fits (one shared GLMM per fold):\n{pool_fold.to_string(index=False)}")
display(cv.groupby('route_stage').agg(
    n=('gene', 'size'), nz_med=('nz', 'median'), mean_z=('mean_z', 'mean'), std_z=('std_z', 'mean'),
    exceed=('cv_naive_exceed', 'mean'), stable=('stable', 'mean')).round(4))

## 1. Fit failure / convergence

In [ ]:
'''
1. Fit failure / convergence
Two distinct things, deliberately separated: (a) genes the deployed engine could not fit at all
(train "excluded"), and (b) genes the engine DID fit on the full cohort but whose per-gene GLMM
fails to converge when refit on a 4/5 subsample. (b) is a stability property invisible in-sample --
each CV fold trains on ~541 of 676 HC -- and it is the quantity the nz_a_max choice hinges on.
'''
fold_nz = fold.merge(train[['gene', 'nz']], on='gene')
fold_nz['nz_bin'] = pd.cut(fold_nz['nz'], bins=NZ_BINS, labels=NZ_LABELS)
by_nz = fold_nz.groupby('nz_bin', observed=True).agg(n_gene=('gene', 'nunique'), fold_ok=('ok', 'mean'))
by_nz['all5_ok'] = cv[cv.route == 'model'].groupby('nz_bin', observed=True)['stable'].mean()

print(f"model-route fold success: {fold['ok'].mean():.4f} ({int(fold['ok'].sum())}/{len(fold)})")
print(f"genes converging in all 5 folds: {cv.loc[cv.route == 'model', 'stable'].mean():.4f}")
print(f"\ntrain-stage routing (deployed engine):\n"
      f"{train.groupby(['route', 'stage'], dropna=False).size().to_string()}")
print(f"\nfold fail_reason:\n{fold.loc[~fold['ok'], 'fail_reason'].fillna('(blank)').value_counts().to_string()}")
display(by_nz.round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
x = np.arange(len(by_nz))
axes[0].plot(x, by_nz['fold_ok'], marker='o', color='#2b5c8f', label='per-fold convergence')
axes[0].plot(x, by_nz['all5_ok'], marker='s', color='#d62828', label='all 5 folds')
axes[0].axhline(0.95, color='black', linewidth=0.8, linestyle='--')
axes[0].axvline(-0.5, color='#81b29a', linewidth=2)
axes[0].set_xticks(x); axes[0].set_xticklabels(by_nz.index, rotation=45, ha='right')
axes[0].set_ylabel('convergence rate'); axes[0].set_xlabel('nz bin (model route only)')
axes[0].set_title(f'Convergence vs nz\n(pool route holds nz < {config.NZ_A_MAX})', fontsize=11, fontweight='bold')
axes[0].legend(frameon=False, fontsize=9)

nfold = cv.loc[cv.route == 'model', 'n_fold_ok'].value_counts().sort_index()
axes[1].bar(nfold.index.astype(int), nfold.values, color='#e07a5f', edgecolor='black')
axes[1].set_yscale('log')
axes[1].set_xlabel('# folds converged (of 5)'); axes[1].set_ylabel('# genes (log)')
axes[1].set_title('Per-gene fold convergence count', fontsize=11, fontweight='bold')
for i, v in zip(nfold.index.astype(int), nfold.values):
    axes[1].text(i, v, str(int(v)), ha='center', va='bottom', fontsize=8)

# tau2 is refit per fold; its spread across folds is a second, continuous stability read-out that
# does not depend on the binary convergence flag.
t2 = fold_nz[fold_nz['ok']].groupby('gene').agg(tau2_cv=('tau2', lambda v: v.std() / max(v.mean(), 1e-9)),
                                                nz=('nz', 'first'))
t2['nz_bin'] = pd.cut(t2['nz'], bins=NZ_BINS, labels=NZ_LABELS)
med = t2.groupby('nz_bin', observed=True)['tau2_cv'].median()
axes[2].bar(np.arange(len(med)), med.values, color='#81b29a', edgecolor='black')
axes[2].set_xticks(np.arange(len(med))); axes[2].set_xticklabels(med.index, rotation=45, ha='right')
axes[2].set_ylabel('median CV of tau2 across folds'); axes[2].set_xlabel('nz bin')
axes[2].set_title('Batch-variance instability vs nz', fontsize=11, fontweight='bold')

plt.tight_layout(); plt.savefig(FIG / 'cv_convergence.png', dpi=150, bbox_inches='tight'); plt.show()

## 2. Held-out Z calibration

In [ ]:
'''
2. Held-out Z calibration
Held-out HC is a true null, so the RQR z must be N(0,1) per gene. std_z is the load-bearing
quantity: >1 inflates the z_flag=3 false-positive rate downstream, <1 costs detection power.
'''
zc = ['mean_z', 'std_z', 'cv_naive_exceed']
print(f"all genes: mean_z={cv.mean_z.mean():+.4f} std_z={cv.std_z.mean():.4f} "
      f"exceed={cv.cv_naive_exceed.mean():.4f}  (targets 0 / 1 / 0.05)")
display(cv.groupby('route_stage')[zc].agg(['size', 'mean', 'median', 'std']).round(4))
display(cv.groupby(['route', 'nz_bin'], observed=True).agg(
    n=('gene', 'size'), mean_z=('mean_z', 'mean'), std_z=('std_z', 'mean'),
    exceed=('cv_naive_exceed', 'mean')).round(4))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
colors = {'model': '#2b5c8f', 'pool': '#e07a5f'}
for r, c in colors.items():
    sub = cv[cv.route == r]
    axes[0, 0].hist(sub['std_z'], bins=np.linspace(0.5, 1.5, 80), alpha=0.55, color=c,
                    label=f'{r} (n={len(sub)}, mean={sub.std_z.mean():.3f})', density=True)
axes[0, 0].axvline(1.0, color='black', linewidth=1, linestyle='--')
axes[0, 0].set_xlabel('std_z (held-out)'); axes[0, 0].set_ylabel('density')
axes[0, 0].set_title('Z scale calibration by route', fontsize=11, fontweight='bold')
axes[0, 0].legend(frameon=False, fontsize=9)

for r, c in colors.items():
    sub = cv[cv.route == r]
    axes[0, 1].hist(sub['mean_z'], bins=np.linspace(-0.4, 0.4, 80), alpha=0.55, color=c, density=True)
axes[0, 1].axvline(0.0, color='black', linewidth=1, linestyle='--')
axes[0, 1].set_xlabel('mean_z (held-out)'); axes[0, 1].set_ylabel('density')
axes[0, 1].set_title('Z location calibration by route', fontsize=11, fontweight='bold')

for r, c in colors.items():
    sub = cv[cv.route == r].groupby('nz_bin', observed=True)['std_z'].mean()
    axes[1, 0].plot(np.arange(len(NZ_LABELS)), sub.reindex(NZ_LABELS).values, marker='o', color=c, label=r)
axes[1, 0].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes[1, 0].set_xticks(np.arange(len(NZ_LABELS))); axes[1, 0].set_xticklabels(NZ_LABELS, rotation=45, ha='right')
axes[1, 0].set_ylabel('mean std_z'); axes[1, 0].set_xlabel('nz bin')
axes[1, 0].set_title('Z scale vs expression depth', fontsize=11, fontweight='bold')
axes[1, 0].legend(frameon=False, fontsize=9)

# The flag rate that actually matters downstream: |z| > z_flag on a true null.
Z_FLAG = 3.0
zs = pickle.load(open(CV / 'cv_zscores.pkl', 'rb'))
flag = pd.Series({g: float(np.mean(np.abs(v[np.isfinite(v)]) > Z_FLAG)) for g, v in zs.items()}, name='flag_rate')
cv = cv.merge(flag, left_on='gene', right_index=True, how='left')
from scipy.stats import norm as _norm
nominal = 2 * (1 - _norm.cdf(Z_FLAG))
for r, c in colors.items():
    sub = cv[cv.route == r]
    axes[1, 1].hist(sub['flag_rate'], bins=np.linspace(0, 0.03, 60), alpha=0.55, color=c,
                    label=f'{r} (mean={sub.flag_rate.mean():.5f})', density=True)
axes[1, 1].axvline(nominal, color='black', linewidth=1, linestyle='--', label=f'nominal {nominal:.5f}')
axes[1, 1].set_xlabel(f'|z| > {Z_FLAG:g} rate on held-out HC'); axes[1, 1].set_ylabel('density')
axes[1, 1].set_title('False-flag rate (true null)', fontsize=11, fontweight='bold')
axes[1, 1].legend(frameon=False, fontsize=8)

plt.tight_layout(); plt.savefig(FIG / 'cv_z_calibration.png', dpi=150, bbox_inches='tight'); plt.show()
print(f"\nfalse-flag rate vs nominal {nominal:.5f}: "
      f"model={cv.loc[cv.route=='model','flag_rate'].mean():.5f}  pool={cv.loc[cv.route=='pool','flag_rate'].mean():.5f}")

## 3. Pooled Z vs N(0,1)

In [ ]:
# 3. Pooled Z distribution vs N(0,1)
# Per-gene summaries can each look fine while the pooled distribution is wrong in the tails, which
# is where z_flag lives. Sampled to keep the QQ tractable.
from scipy.stats import norm as _norm

rng = np.random.default_rng(0)
SAMPLE_PER_GENE = 40
pool_z = {r: [] for r in ['model', 'pool']}
route_of = cv.set_index('gene')['route'].to_dict()
for g, v in zs.items():
    r = route_of.get(g)
    if r is None:
        continue
    v = v[np.isfinite(v)]
    if len(v):
        pool_z[r].append(rng.choice(v, min(SAMPLE_PER_GENE, len(v)), replace=False))
pool_z = {r: np.concatenate(v) for r, v in pool_z.items() if v}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
grid = np.linspace(-5, 5, 400)
for r, c in {'model': '#2b5c8f', 'pool': '#e07a5f'}.items():
    z = pool_z[r]
    axes[0].hist(z, bins=200, range=(-5, 5), density=True, histtype='step', linewidth=1.5, color=c,
                 label=f'{r} (n={len(z):,}, sd={z.std():.3f})')
    q = np.linspace(0.0005, 0.9995, 2000)
    axes[1].plot(_norm.ppf(q), np.quantile(z, q), color=c, linewidth=1.5, label=r)
axes[0].plot(grid, _norm.pdf(grid), color='black', linewidth=1.2, linestyle='--', label='N(0,1)')
axes[0].set_yscale('log'); axes[0].set_xlabel('z'); axes[0].set_ylabel('density (log)')
axes[0].set_title('Pooled held-out Z vs N(0,1)\n(log scale: tail behaviour)', fontsize=11, fontweight='bold')
axes[0].legend(frameon=False, fontsize=9)
axes[1].plot([-4, 4], [-4, 4], color='black', linewidth=1, linestyle='--')
for zf in (-3, 3):
    axes[1].axvline(zf, color='#81b29a', linewidth=0.8)
axes[1].set_xlabel('theoretical N(0,1) quantile'); axes[1].set_ylabel('observed quantile')
axes[1].set_title('QQ plot (green = z_flag=3)', fontsize=11, fontweight='bold')
axes[1].legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig(FIG / 'cv_z_qq.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. SHASH calibration

In [ ]:
'''
4. SHASH calibration gain
Genes whose held-out RQR carries real skew/kurtosis break the N(0,1) assumption downstream FDR
relies on. calibration.py fits a per-gene SHASH and re-standardises; this checks the correction
earns its place rather than assuming it.
'''
ok_sh = cv[cv['cv_shash_ok']].copy()
ok_sh['exceed_gain'] = ok_sh['cv_naive_exceed'] - ok_sh['cv_shash_exceed']
ok_sh['fdr_gain'] = ok_sh['cv_naive_fdr_reject_rate'] - ok_sh['cv_corr_fdr_reject_rate']
print(f"shash_ok: {len(ok_sh)}/{len(cv)}")
display(ok_sh.groupby('route').agg(
    naive_exceed=('cv_naive_exceed', 'mean'), shash_exceed=('cv_shash_exceed', 'mean'),
    naive_fdr=('cv_naive_fdr_reject_rate', 'mean'), corr_fdr=('cv_corr_fdr_reject_rate', 'mean'),
    raw_skew=('cv_raw_skew', 'median'), corr_skew=('cv_corrected_skew', 'median'),
    raw_kurt=('cv_raw_kurtosis', 'median'), corr_kurt=('cv_corrected_kurtosis', 'median')).round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(ok_sh['cv_raw_skew'], ok_sh['cv_corrected_skew'], s=3, alpha=0.12, color='#2b5c8f')
axes[0].axhline(0, color='black', linewidth=0.8); axes[0].plot([-3, 3], [-3, 3], 'k--', linewidth=0.8)
axes[0].set_xlabel('raw skew'); axes[0].set_ylabel('SHASH-corrected skew')
axes[0].set_title('Skew correction', fontsize=11, fontweight='bold')
axes[1].scatter(ok_sh['cv_raw_kurtosis'], ok_sh['cv_corrected_kurtosis'], s=3, alpha=0.12, color='#e07a5f')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('raw excess kurtosis'); axes[1].set_ylabel('corrected excess kurtosis')
axes[1].set_title('Kurtosis correction', fontsize=11, fontweight='bold')
axes[2].scatter(ok_sh['cv_raw_kurtosis'].abs(), ok_sh['fdr_gain'], s=3, alpha=0.12, color='#81b29a')
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_xlabel('|raw excess kurtosis|'); axes[2].set_ylabel('naive - corrected FDR reject rate')
axes[2].set_title('FDR gain vs kurtosis', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.savefig(FIG / 'cv_shash.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. PPC (moments)

In [ ]:
'''
5. PPC on held-out folds (moment level)
cv_stats already carries obs/pred mean, variance and zero fraction. Note pred_var is
E[Var(y_i|x_i)] and omits Var(mu_i) across samples while obs_var is the total across-sample
variance -- the two are not on the same footing, so the diagonal is NOT a calibration reference
here (sec. 6 does the replicate-based check that is). Mean and zero fraction ARE comparable.
'''
p = cv.dropna(subset=['cv_obs_mean', 'cv_pred_mean']).copy()
p['log_var_ratio'] = np.log(p['cv_pred_var'] / np.maximum(p['cv_obs_var'], 1e-12))
print(f"median |zero_diff|: {p['cv_zero_diff'].abs().median():.4f}")
print(f"median mean_rel_err: {p['cv_mean_rel_err'].median():+.4f}")
print(f"median log(pred_var/obs_var): {p['log_var_ratio'].median():+.4f}  (see caveat above)")
display(p.groupby('route').agg(
    zero_diff=('cv_zero_diff', 'median'), mean_rel_err=('cv_mean_rel_err', 'median'),
    log_var_ratio=('log_var_ratio', 'median'), ll_mean=('cv_ll_mean', 'median')).round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(p['cv_obs_mean'], p['cv_pred_mean'], s=3, alpha=0.12, color='#2b5c8f')
lim = [max(p['cv_obs_mean'].min(), 1e-3), p['cv_obs_mean'].max()]
axes[0].plot(lim, lim, 'k--', linewidth=1); axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_xlabel('observed mean'); axes[0].set_ylabel('predicted mean')
axes[0].set_title('Mean calibration (held-out)', fontsize=11, fontweight='bold')
axes[1].scatter(p['cv_obs_zero_frac'], p['cv_pred_zero_frac'], s=3, alpha=0.12, color='#e07a5f')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[1].set_xlabel('observed zero fraction'); axes[1].set_ylabel('predicted zero fraction')
axes[1].set_title('Zero-inflation calibration', fontsize=11, fontweight='bold')
for r, c in {'model': '#2b5c8f', 'pool': '#e07a5f'}.items():
    sub = p[p.route == r].groupby('nz_bin', observed=True)['cv_zero_diff'].median()
    axes[2].plot(np.arange(len(NZ_LABELS)), sub.reindex(NZ_LABELS).values, marker='o', color=c, label=r)
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_xticks(np.arange(len(NZ_LABELS))); axes[2].set_xticklabels(NZ_LABELS, rotation=45, ha='right')
axes[2].set_ylabel('median pred - obs zero fraction'); axes[2].set_xlabel('nz bin')
axes[2].set_title('Zero-inflation bias vs nz', fontsize=11, fontweight='bold')
axes[2].legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig(FIG / 'cv_ppc_moments.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. PPC (replicates)

In [ ]:
'''
6. PPC with posterior-predictive replicates (cached)
The moment scatter above cannot judge variance (pred_var is not the same quantity as obs_var).
Replicates can: draw y_rep from each held-out sample's own fitted (mu, alpha, tau2), so rep_var
carries the same across-sample mu variation the observed data has. Bayesian p near 0 or 1 flags a
gene the model cannot reproduce. Stratified subsample by nz -- 20k genes x reps is not worth the
runtime, and the cached CSV is reused on re-run.
'''
from scipy.stats import nbinom

PPC_CACHE = CV / 'cv_ppc_replicates.csv'
PPC_N_REPS = 200
PPC_N_GENES = 3000

def simulate_mixed(mu, alpha, tau2, n_reps, rng):
    if np.all(tau2 < 1e-12):
        mu_b = np.broadcast_to(mu, (n_reps, len(mu)))
    else:
        b = rng.normal(0.0, np.sqrt(np.maximum(tau2, 0.0)), size=(n_reps, len(mu)))
        mu_b = np.clip(mu * np.exp(b), 1e-8, 1e10)
    n_ = np.broadcast_to(1.0 / np.maximum(alpha, 1e-8), (n_reps, len(mu)))
    return rng.negative_binomial(n_, np.clip(n_ / (n_ + mu_b), 1e-10, 1 - 1e-10))

if PPC_CACHE.exists():
    ppc = pd.read_csv(PPC_CACHE)
else:
    from tqdm.auto import tqdm
    ppcd = pickle.load(open(CV / 'cv_ppc.pkl', 'rb'))
    # Stratify on nz deciles, not NZ_BINS: those start at the pooling cutoff and would drop
    # every pool-route gene, which is the route that most needs checking.
    strata = pd.qcut(cv['nz'].rank(method='first'), 10, labels=False)
    pick = (cv.assign(_s=strata).groupby(['route', '_s'], observed=True)['gene']
              .apply(lambda s: s.sample(min(len(s), PPC_N_GENES // 20), random_state=0)))
    rows = []
    for i, g in enumerate(tqdm(pick.values, desc='ppc replicates')):
        d = ppcd.get(g)
        if d is None:
            continue
        y, mu, alpha, tau2 = d['y'], d['mu'], d['alpha'], np.atleast_1d(d['tau2'])
        rng = np.random.default_rng(7000 + i)
        yr = simulate_mixed(mu, alpha, tau2, PPC_N_REPS, rng)
        rows.append({'gene': g,
                     'p_mean': float(np.mean(yr.mean(1) >= y.mean())),
                     'p_var': float(np.mean(yr.var(1) >= y.var())),
                     'p_zero': float(np.mean((yr == 0).mean(1) >= (y == 0).mean())),
                     'p_max': float(np.mean(yr.max(1) >= y.max()))})
    ppc = pd.DataFrame(rows)
    ppc.to_csv(PPC_CACHE, index=False)

ppc = ppc.merge(cv[['gene', 'route', 'stage', 'nz', 'nz_bin', 'stable', 'std_z']], on='gene')
pcols = ['p_mean', 'p_var', 'p_zero', 'p_max']
print(f"genes with replicates: {len(ppc)}")
display(ppc.groupby('route')[pcols].agg(['median', lambda v: (v < 0.05).mean(), lambda v: (v > 0.95).mean()])
        .rename(columns={'<lambda_0>': 'frac<0.05', '<lambda_1>': 'frac>0.95'}).round(4))

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, c in zip(axes, pcols):
    for r, col in {'model': '#2b5c8f', 'pool': '#e07a5f'}.items():
        sub = ppc[ppc.route == r]
        ax.hist(sub[c], bins=25, range=(0, 1), alpha=0.55, color=col, density=True, label=r)
    ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel(c); ax.set_ylabel('density')
    ax.set_title(f'PPC {c}\n(flat = well calibrated)', fontsize=11, fontweight='bold')
axes[0].legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig(FIG / 'cv_ppc_replicates.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Fold stability vs Z calibration

In [ ]:
'''
7. Does fold instability damage the deployed Z?
The nz_a_max=25 decision rests on this. Genes below nz~50 often fail to converge in some folds
(sec. 1), which argues the individual fit is not trustworthy there. But the deployed artefact is
the Z, not the coefficient vector -- so compare calibration between stable and unstable genes at
matched nz. If unstable genes are still calibrated, fold non-convergence is a coefficient-level
property that does not propagate, and pooling them would buy nothing.
'''
mr = cv[cv.route == 'model'].copy()
cmp_tbl = mr.groupby(['nz_bin', 'stable'], observed=True).agg(
    n=('gene', 'size'), mean_z=('mean_z', 'mean'), std_z=('std_z', 'mean'),
    exceed=('cv_naive_exceed', 'mean'), flag_rate=('flag_rate', 'mean'),
    zero_diff=('cv_zero_diff', 'median')).round(4)
display(cmp_tbl)
lo = mr[mr.nz < 100]
print(f"nz<100  stable n={int(lo.stable.sum())} std_z={lo.loc[lo.stable,'std_z'].mean():.4f} | "
      f"unstable n={int((~lo.stable).sum())} std_z={lo.loc[~lo.stable,'std_z'].mean():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
w = 0.38; x = np.arange(len(NZ_LABELS))
for k, (st, col) in enumerate({True: '#2b5c8f', False: '#d62828'}.items()):
    v = mr[mr.stable == st].groupby('nz_bin', observed=True)['std_z'].mean().reindex(NZ_LABELS)
    axes[0].bar(x + (k - 0.5) * w, v.values, w, color=col, edgecolor='black',
                label='all 5 folds' if st else 'some fold failed')
axes[0].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xticks(x); axes[0].set_xticklabels(NZ_LABELS, rotation=45, ha='right')
axes[0].set_ylim(0.9, 1.1); axes[0].set_ylabel('mean std_z'); axes[0].set_xlabel('nz bin')
axes[0].set_title('Z calibration by fold stability\n(matched nz)', fontsize=11, fontweight='bold')
axes[0].legend(frameon=False, fontsize=9)

axes[1].scatter(mr.loc[mr.stable, 'nz'], mr.loc[mr.stable, 'std_z'], s=3, alpha=0.10, color='#2b5c8f', label='all 5 folds')
axes[1].scatter(mr.loc[~mr.stable, 'nz'], mr.loc[~mr.stable, 'std_z'], s=6, alpha=0.45, color='#d62828', label='some fold failed')
axes[1].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes[1].axvline(config.NZ_A_MAX, color='#81b29a', linewidth=1.5, label=f'nz_a_max={config.NZ_A_MAX}')
axes[1].set_xscale('log'); axes[1].set_ylim(0.6, 1.4)
axes[1].set_xlabel('nz (log)'); axes[1].set_ylabel('std_z')
axes[1].set_title('std_z vs nz', fontsize=11, fontweight='bold')
axes[1].legend(frameon=False, fontsize=9, markerscale=3)
plt.tight_layout(); plt.savefig(FIG / 'cv_stability_vs_calibration.png', dpi=150, bbox_inches='tight'); plt.show()

## 8. Route summary

In [ ]:
'''
8. Route comparison and summary table
The pool route had never been run on real HC before this engine, so its held-out behaviour is
reported next to the model route rather than assumed equivalent.
'''
summ_rows = []
for r in ['model', 'pool']:
    s = cv[cv.route == r]
    pr = ppc[ppc.route == r]
    summ_rows.append({
        'route': r, 'n_genes': len(s), 'nz_median': s['nz'].median(),
        'mean_z': s['mean_z'].mean(), 'std_z': s['std_z'].mean(),
        'exceed_95': s['cv_naive_exceed'].mean(), 'flag_rate_z3': s['flag_rate'].mean(),
        'zero_diff': s['cv_zero_diff'].median(), 'll_mean': s['cv_ll_mean'].median(),
        'ppc_p_var_extreme': float(((pr['p_var'] < 0.05) | (pr['p_var'] > 0.95)).mean()),
        'ppc_p_zero_extreme': float(((pr['p_zero'] < 0.05) | (pr['p_zero'] > 0.95)).mean()),
    })
summary = pd.DataFrame(summ_rows).set_index('route')
display(summary.round(5))
summary.to_csv(CV / 'cv_route_summary.csv')

print(f"targets: mean_z 0 | std_z 1 | exceed_95 {TARGET['exceed']} | "
      f"flag_rate_z3 {2 * (1 - _norm.cdf(3.0)):.5f}")
print(f"\nengine: nz_a_max={config.NZ_A_MAX} tau2_max={config.FIT_PARAMS['tau2_max']} "
      f"pcis_cut={config.FIT_PARAMS['pcis_cut']} min_hc_batch={config.MIN_HC_BATCH_SIZE}")
print(f"excluded genes (no usable fit): {int((~train['ok']).sum())} / {len(train)}")
print(f"saved -> {CV / 'cv_route_summary.csv'}, figures in {FIG}")